# Notebook 02 — Why Is My Monthly Report Instant But My Colleague's Takes Forever?
## Same Query, Same Data — The Hidden Impact of Data Loading Order

**What you'll learn**: The order in which data is loaded into a table directly affects how fast your queries run — even if the data is identical.

| | TRANSACTIONS (Well-Loaded) | TRANSACTIONS_UNORDERED (Poorly-Loaded) |
|---|---|---|
| **How data was loaded** | In date order (like a bank ledger) | Randomly shuffled |
| **Your monthly report** | Snowflake skips 97% of the data | Snowflake reads EVERYTHING |
| **Speed** | Seconds | 10-20x slower |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Run the SAME Query on Both Tables

**Scenario**: "Pull all transactions from November 2024" — a standard monthly reconciliation report.

### On the ORDERED Table (Best Case)

In [ ]:
-- Disable result cache so we get true execution metrics
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

In [ ]:
-- BEST CASE: Naturally ordered table — Snowflake prunes most partitions
SELECT
    transaction_type,
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1, 2
ORDER BY total_amount DESC;

**Open the Query Profile** → Note: Partitions scanned ~2-3% of total. Fast.

### On the UNORDERED Table (Worst Case)

In [ ]:
-- WORST CASE: Same query, same data, but table was loaded in random order
SELECT
    transaction_type,
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount
FROM TRANSACTIONS_UNORDERED
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
GROUP BY 1, 2
ORDER BY total_amount DESC;

**Open the Query Profile** → Note: Partitions scanned ~95-100% of total. Full scan!

---
## Step 3: Side-by-Side Metrics

In [ ]:
-- Compare the two queries head-to-head (using INFORMATION_SCHEMA)
-- Note: partitions_scanned/partitions_total are only in ACCOUNT_USAGE;
-- INFORMATION_SCHEMA provides bytes_scanned and timing metrics.
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_UNORDERED%' THEN 'UNORDERED (shuffled)'
        ELSE 'ORDERED (chronological)'
    END AS table_version,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    compilation_time,
    execution_time
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -10, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%transaction_type%channel%2024-11-01%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Key Takeaways

| | Well-Loaded Table | Poorly-Loaded Table |
|---|---|---|
| **Your same report** | Runs in seconds | 10-20x slower |
| **Data scanned** | ~2-3% (Snowflake skips the rest) | ~100% (reads everything) |
| **Why** | Dates are packed together → Snowflake knows where to look | Dates are scattered everywhere → Snowflake must check everything |
| **Cost to you** | Free — just how the data was loaded | You pay for a full scan on every single query |

**What this means for you as an analyst**:  
If your monthly/quarterly/yearly reports feel slow, the problem might not be your SQL — it could be how the data was loaded. This is worth raising with your data engineering or platform team.

**Banking examples where this matters**:  
- Daily reconciliation, monthly statements, regulatory reporting → naturally fast when data is loaded in date order
- Same queries on a poorly-loaded table → expensive and slow, even with identical data

**💡 What to tell your platform team**: *"Our TRANSACTIONS table is loaded chronologically — that's great for date-range reports. Please keep it that way."*

**Next** → [Notebook 03 — Why Is My Customer Lookup So Slow?](./Notebook_03_Clustering.ipynb)